In [ ]:
# @title Set ONLINE = False if running locally
ONLINE = True

In [ ]:
# Install necessary libraries
!pip install pandas scikit-surprise

import pandas as pd
from surprise import SVD , SVDpp
from surprise.model_selection import train_test_split , GridSearchCV
from surprise import Dataset, Reader
from surprise.accuracy import rmse
import numpy
import gdown
import os

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

# Load the training data
data_train = pd.read_csv('data_train.csv')

# Load the sample submission data
sample_submission = pd.read_csv('sampleSubmission.csv')

In [ ]:
# Extract user_id and item_id from the 'Id' column
data_train['user_id'] = data_train['Id'].apply(lambda x: int(x.split('_')[0][1:]))
data_train['item_id'] = data_train['Id'].apply(lambda x: int(x.split('_')[1][1:]))
data_train['rating'] = data_train['Prediction']

# Prepare the data for the surprise library
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(data_train[['user_id', 'item_id', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2,random_state=42)

In [ ]:
# Baseline algorithm using SVD
algo_svd = SVD()
algo_svd.fit(trainset)

# Predict ratings for the testset
predictions_svd = algo_svd.test(testset)

# Calculate RMSE
rmse_value_svd = rmse(predictions_svd, verbose=True)

In [ ]:
# Baseline algorithm using SVDpp
algo_svdpp = SVDpp()
algo_svdpp.fit(trainset)

# Predict ratings for the testset
predictions_svdpp = algo_svdpp.test(testset)

# Calculate RMSE
rmse_value_svdpp = rmse(predictions_svdpp, verbose=True)

In [ ]:
# Use SVD and SVDpp algorithms and perform GridSearch for parameter tuning as follows :
"""
param_grid = {
    'n_factors': [1,2,5,10,20,50,75,100,..1000...],
    'n_epochs': [10,20,30,...60,...],
    'lr_all': [0.001,0.002,0.005,0.01],
    'reg_all': [ 0.1],
     'biased': [True,False]

}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

# Best SVD model
algo = gs.best_estimator['rmse']

print("Best parameters:", gs.best_params['rmse'])
print("Best RMSE:", gs.best_score['rmse'])

"""
# The best parameters found using GridSearch were the following :
algo_svd_finetuned = SVD(n_factors=75, n_epochs=85, lr_all=0.005, reg_all=0.1, biased = True)
algo_svd_finetuned.fit(trainset)

# Predict ratings for the testset
predictions_svd_finetuned = algo_svd_finetuned.test(testset)

# Calculate RMSE
rmse_value_svd = rmse(predictions_svd_finetuned, verbose=True)

In [ ]:
# Fine-tuning of SVDpp
algo_svdpp_finetuned = SVDpp(n_factors=75, n_epochs=85, lr_all=0.005, reg_all=0.1)
algo_svdpp_finetuned.fit(trainset)

# Predict ratings for the testset
predictions_svdpp_finetuned = algo_svdpp_finetuned.test(testset)

# Calculate RMSE
rmse_value_svdpp_finetuned = rmse(predictions_svdpp_finetuned, verbose=True)

In [ ]:
# Extract user_id and item_id from the 'Id' column in the sample submission file
sample_submission['user_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[0][1:]))
sample_submission['item_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[1][1:]))

# Function to make submissions
def make_submission(algo, submission_df, output_path):
    def predict_rating(user_id, item_id):
        prediction = algo.predict(user_id, item_id).est
        # Clip the prediction to the desired range
        clipped_prediction = numpy.clip(prediction, 1, 5)
        return clipped_prediction

    submission_df['Prediction'] = submission_df.apply(lambda row: predict_rating(row['user_id'], row['item_id']), axis=1)
    submission_df[['Id', 'Prediction']].to_csv(output_path, index=False)

# Choose the algorithm to submit
chosen_algo = algo_svd_finetuned # You can change this to algo_svd, algo_svdpp, or algo_svdpp_finetuned

# Make the submission
make_submission(chosen_algo, sample_submission, 'my_submission_svdpp_finetuned.csv')